In [1]:

# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

Cloning into '/content/Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 370, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 370 (delta 94), reused 97 (delta 39), pack-reused 190 (from 1)
Receiving objects: 100% (370/370), 16.38 MiB | 2.86 MiB/s, done.
Resolving deltas: 100% (185/185), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [2]:
# =============================
# CONTROLLED VERBOSITY
# =============================

# Disable HF download progress bars BEFORE importing anything HF-related

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [3]:
# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.6 MB/s eta 0:00:00


In [4]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

import importlib

import captioning
importlib.reload(captioning)
from captioning import run_captioning


import training_evaluation
importlib.reload(training_evaluation)
from training_evaluation import run_training

In [5]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import random
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging


# Silence transformers & HF logs (keep only errors)
transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [6]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [7]:
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

In [8]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

'/content/Project_Generative_AI_for_Data_Augmentation'

In [9]:
# control flags.
RUN_CAPTIONING = True
RUN_TEXT_VARIATION = True
RUN_IMAGE_GENERATION = True
RUN_TRAINING = True

In [10]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

100%|██████████| 792M/792M [00:55<00:00, 14.3MB/s]
100%|██████████| 19.2M/19.2M [00:03<00:00, 5.87MB/s]


Train size: 3680
Test size: 3669


In [11]:
# extract labels
labels = dataset_train._labels
indices = np.arange(len(dataset_train))

# perform stratified split
train_small_idx, _ = train_test_split(
    indices,
    train_size=0.30,
    stratify=labels,
    random_state=42
)

SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

np.save(os.path.join(SPLIT_DIR, "train_small_indices.npy"), train_small_idx)

In [12]:
# # run for ALL
# # create subset dataset from training set
dataset_train_small = Subset(dataset_train, train_small_idx)

In [13]:
# run for 10
dataset_train_small_10 = Subset(dataset_train, train_small_idx[:10])

# Captioning

In [14]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [15]:
if RUN_CAPTIONING:

  device = "cuda" if torch.cuda.is_available() else "cpu"

  processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

  model = Blip2ForConditionalGeneration.from_pretrained(
      "Salesforce/blip2-opt-2.7b",
      torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32 # to reduce GPU memory usage
  )

  model.to(device)
  model.eval()



  captions_dict = run_captioning(
      # dataset_train_small=dataset_train_small, # FOR ALL
      dataset_train_small=dataset_train_small_10, # FOR 10
      model=model,
      processor=processor,
      device=device,
      output_path=CAPTION_PATH
  )

  del model
  del processor
  clear_gpu()
  !nvidia-smi

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 10/10 [00:08<00:00,  1.16it/s]


Full caption generation completed.
GPU memory cleared.
Mon Feb 23 15:37:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             53W /  400W |     546MiB /  40960MiB |      1%      Default |
|                                         |                        |             Disa

In [17]:
# check date time of last change of given file
!stat "$CAPTION_PATH"

  File: /content/Project_Generative_AI_for_Data_Augmentation/data/captions/captions_train_small_10.json
  Size: 1815      	Blocks: 8          IO Block: 4096   regular file
Device: 36h/54d	Inode: 1839684     Links: 1
Access: (0644/-rw-r--r--)  Uid: (    0/    root)   Gid: (    0/    root)
Access: 2026-02-23 15:31:47.056194083 +0000
Modify: 2026-02-23 15:37:25.605614065 +0000
Change: 2026-02-23 15:37:25.605614065 +0000
 Birth: 2026-02-23 15:31:47.056194083 +0000


# Text variation

## FLAN-T5-Large Model



In [18]:
import text_variation_flan_large
importlib.reload(text_variation_flan_large)
from text_variation_flan_large import run_text_variation as run_flan_large

In [19]:
MAX_ITEMS = 10

In [20]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [21]:
run_flan_large(
    caption_file=CAPTION_PATH,
    max_items=MAX_ITEMS
)

  0%|          | 0/10 [00:00<?, ?it/s]


Original: a pomeranian dog my dog is sitting on the bed
Generated: ['three-leggi', 'little german male standing dog by your apartment room bedroom balcony enjoying nature walks through some countryside as part of its annual activity on', 'Two pets lying naked. Some lying off leh side and are going towards. Three pictures then there with our pomelized German dogs one, two with two different dogs']


 10%|█         | 1/10 [00:02<00:26,  2.97s/it]


Original: a pomeranian dog this dog is sitting on the bed
Generated: ["An inefably adorable canter that wants his place, not gets anywhere where her new human loves at their pet cat peaked chair under bedrail that is right at door when puppy' the", 'the brown and the red puppies attracted all attention for months so decided we want our young dog the PooMd with collar as they all thought about who stole something like we just were.he', 'two of the dogs sleep and they play for 0 degrees to each dm in range the others sit at all types.a small female baby sitting up, in close close']

Original: a havanese dog is sitting on a tennis court
Generated: ['portrait view shows oax playing double fault football off on tennis with dogs under two feet deep underneath the players is doing their sports playing back field,the top four', 'It can sit in or walk - or standing for it?! And with those legs', 'an exotic looking horse or cow looks toward you by it court.eg black cow standing by by dog holdin

 20%|██        | 2/10 [00:05<00:23,  2.88s/it]

...


Original: a saint bernard dog is standing in the snow
Generated: ['it09709 is one black lab doing sit stand k. by white dog white house for an act against dogs from war or to save people as good to others but it the dog of', 'The Saint Albertdog outside walking beside thelfing on the road where it got her foot in that day', 'there standing and kiss this grey can welfer sitting there while looking through camera view over mountains as someone smile by standing while wearing these festive vestions near mountains of christmas time inside and skiing with snowboard']

Original: a havanese dog is standing on a wooden staircase
Generated: ['two beautiful looking and cub obsessed hunting donghire on stone at old castle castle from city.', 'man sitting near old dog under glass roof sitting down side against light colored white sky on snow in dark dayroom outdoors with sky reflecting water light colors all dark shades inside inside frame the white space dividing', "Someone sitted around to insp

 90%|█████████ | 9/10 [00:20<00:02,  2.21s/it]


Original: a havanese dog is sitting on the steps
Generated: ['There must not get as dirty outside these photos were taken outside, the Haver dog does smell very much similar color as their eyes with color the black or golden that make it as pure dark the dogs', 'People are out and dogs that were taken at petstore were all outside near giant hay field covered gates that stood for half. A lady had told herself it could always snow with some very sunny morning', 'in dark forest with clouds visible white bark']

Original: a persian cat is looking at the camera with an angry expression
Generated: ['another cute Asian black or green hairless creature lies looking gruis and sick to viewer with the big white spot showing where that scratch on will land him on foot and hissed mouth and grudge', '', 'The kitten was getting away or not taking in time last day or how this girl did it on paper or not this girl was sitting looking']


100%|██████████| 10/10 [00:23<00:00,  2.35s/it]


Original: a persian cat is looking angry
Generated: ['angry cat has long and tufts to his black ears on . the man can find to where pet them too well also that one man might consider her ugly instead they are small', 'here looks in love like it ?ll always live away! but some things matter', 'white big perega cat that says that all is done to keep your money nice for today,']


In [22]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Mon Feb 23 15:41:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             53W /  400W |     548MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

FLAN-T5-Large was evaluated as a candidate model for caption rewriting.

However, qualitative analysis showed several consistent issues:

- **poor semantic preservation:** generated output often drifted away from original meaning

- **hallucinations:** frequent introduction of unrelated object, people, or scenes

Due to these limitations, FLAN-T5-Large was deemed unsuitable for controlled data augmentation.

## FLAN-T5-XL Model

In [23]:
from text_variation_flan_xl import run_text_variation as run_flan_xl

run_flan_xl(
    caption_file=CAPTION_PATH,
    max_items=10
)

  0%|          | 0/10 [00:00<?, ?it/s]


Original: a pomeranian dog my dog is sitting on the bed
Generated: ['a pomeranian dog my pomeranian is sitting on the bed', 'a pomeranian dog my dog is sitting on the bed', 'a pomeranian dog my pomeranian dog is sitting on the bed']


 10%|█         | 1/10 [00:01<00:17,  1.93s/it]


Original: a pomeranian dog this dog is sitting on the bed
Generated: ['a pomeranian dog this pomeranian is sitting on the bed', 'a pomeranian dog this pomeranian dog is sitting on the bed', 'a pomeranian dog this pomeranian is sitting on the bed .']

Original: a havanese dog is sitting on a tennis court
Generated: ['A Havanese dog is sitting on a tennis court.', 'A Havanese dog sits on a tennis court.', 'A Havanese dog is sitting on a tennis court']


 20%|██        | 2/10 [00:03<00:13,  1.67s/it]

...


Original: a saint bernard dog is standing in the snow
Generated: ['a saint bernard dog is standing in the snow', 'a saint bernard is standing in the snow', 'a saint bernard dog standing in the snow']

Original: a havanese dog is standing on a wooden staircase
Generated: ['A Havanese dog is standing on a wooden staircase.', 'A Havanese dog is standing on a wooden staircase', 'A Havanese is standing on a wooden staircase.']


 90%|█████████ | 9/10 [00:13<00:01,  1.36s/it]


Original: a havanese dog is sitting on the steps
Generated: ['A Havanese dog is sitting on the steps.', 'A Havanese dog sits on the steps.', 'A Havanese dog is sitting on the steps']

Original: a persian cat is looking at the camera with an angry expression
Generated: ['a persian cat is looking at the camera with an angry expression', 'a persian cat is looking at the camera with an angry expression .', 'a persian cat is looking at the camera with an angry expression.']


100%|██████████| 10/10 [00:14<00:00,  1.47s/it]


Original: a persian cat is looking angry
Generated: ['a persian cat looks angry', 'a persian cat is looking angry', 'a persian cat is angry']


In [24]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Mon Feb 23 15:42:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             53W /  400W |     548MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

FLAN-T5-XL produced grammatically correct and semantically faithful rewrites.

However, the generated variations showed very low lexical diversity, often resulting in near-duplicate sentences with only minor wording or punctuation changes.

Since the goal of this stage is meaningful data augmentation, higher variation diversity was required.

## Mistral 7B Instruct Model

In [25]:
from text_variation_mistral import run_text_variation as run_mistral

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAPTIONS_DIR = os.path.join(DATA_DIR, "captions")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")


os.makedirs(TEXT_VARIATIONS_DIR, exist_ok=True)

CAPTION_FILE = os.path.join(
    CAPTIONS_DIR,
    "captions_train_small_10.json"
)

TEXT_VARIATION_FILE = os.path.join(
    TEXT_VARIATIONS_DIR,
    "text_variations_train_small_10.json"
)

if RUN_TEXT_VARIATION:

    run_mistral(
        caption_file = CAPTION_FILE,
        output_file= TEXT_VARIATION_FILE,
        max_items=10
    )

100%|██████████| 10/10 [00:46<00:00,  4.61s/it]

Mistral text variations saved.


In [26]:
!stat $TEXT_VARIATION_FILE

  File: /content/Project_Generative_AI_for_Data_Augmentation/data/text_variations/text_variations_train_small_10.json
  Size: 4631      	Blocks: 16         IO Block: 4096   regular file
Device: 36h/54d	Inode: 1839713     Links: 1
Access: (0644/-rw-r--r--)  Uid: (    0/    root)   Gid: (    0/    root)
Access: 2026-02-23 15:31:47.088197160 +0000
Modify: 2026-02-23 15:44:24.540773389 +0000
Change: 2026-02-23 15:44:24.540773389 +0000
 Birth: 2026-02-23 15:31:47.088197160 +0000


In [27]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Mon Feb 23 15:46:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             53W /  400W |     556MiB /  40960MiB |     13%      Default |
|                                         |                        |             Disabled |
+---------------------------

# Caption Selection

In [28]:
from src.image_generation import (CaptionSelector, SyntheticImageGenerator)
import json

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [29]:
with open(TEXT_VARIATION_FILE, "r") as f:
    text_variations = json.load(f)

In [30]:
selector = CaptionSelector()

selected_data = {}

for idx, data in text_variations.items():

    class_name = data["class_name"]
    original_captions = data["original_captions"]
    generated_captions = data["generated_captions"]

    selected_generated = selector.select_top_captions(
        original_captions,
        generated_captions,
        top_k=2
    )

    selected_data[idx] = {
    "class_name": class_name,
    "original_captions": original_captions,
    "selected_generated_captions": selected_generated
}

In [31]:
selected_data

{'3037': {'class_name': 'Pomeranian',
  'original_captions': ['a pomeranian dog my dog is sitting on the bed',
   'a pomeranian dog this dog is sitting on the bed'],
  'selected_generated_captions': ['The Pomeranian breed of dog is occupying the bed where I am sitting.',
   'My Pomeranian dog is seated on the bed.']},
 '2635': {'class_name': 'Havanese',
  'original_captions': ['a havanese dog is sitting on a tennis court',
   'a havanese dog is sitting on the tennis court'],
  'selected_generated_captions': ['A Havanese dog is positioned on a tennis court.',
   'The Havanese dog is situated on a tennis court.']},
 '498': {'class_name': 'British Shorthair',
  'original_captions': ['a british shorthair cat is sitting in a box',
   'a british shorthair cat is sitting in a cardboard box'],
  'selected_generated_captions': ['In a cardboard box resides a British Shorthair cat.',
   'A British Shorthair cat occupies a cardboard box.']},
 '1449': {'class_name': 'Samoyed',
  'original_captions'

# Image Generation

In [32]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

CUDA available: True
Device count: 1


In [33]:
if RUN_IMAGE_GENERATION:
        generator = SyntheticImageGenerator()

        metadata = generator.generate_images(
                selected_data = selected_data,
                output_dir = os.path.join(DATA_DIR, "synthetic/images"),
                checkpoint_file = os.path.join(DATA_DIR, "synthetic/generation_checkpoint.json"),
                final_metadata_file = os.path.join(DATA_DIR, "synthetic/generation_metadata_final.json"),
                batch_size=4
        )

        del generator
        clear_gpu()
        !nvidia-smi

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2213: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(
  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:04<00:36,  4.01s/it]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/20 [00:00<?, ?it/s]

 90%|█████████ | 9/10 [00:30<00:03,  3.26s/it]

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:33<00:00,  3.36s/it]


GPU memory cleared.
Mon Feb 23 15:48:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             71W /  400W |     642MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

In [35]:
# check date time last edit
img_file = os.path.join(DATA_DIR, "synthetic/images/1403_0.png")
!stat $img_file

  File: /content/Project_Generative_AI_for_Data_Augmentation/data/synthetic/images/1403_0.png
  Size: 476949    	Blocks: 936        IO Block: 4096   regular file
Device: 36h/54d	Inode: 5242946     Links: 1
Access: (0644/-rw-r--r--)  Uid: (    0/    root)   Gid: (    0/    root)
Access: 2026-02-23 15:48:36.546931379 +0000
Modify: 2026-02-23 15:48:36.649941254 +0000
Change: 2026-02-23 15:48:36.649941254 +0000
 Birth: 2026-02-23 15:48:36.546931379 +0000


In [ ]:
'''import shutil

shutil.make_archive(
    "synthetic_images",
    'zip',
    os.path.join(DATA_DIR, "synthetic/images")
)'''

In [ ]:
'''from google.colab import files
files.download("synthetic_images.zip")'''

In [ ]:
'''import os
import datetime

file_path = TEXT_VARIATION_FILE

timestamp = os.path.getmtime(file_path)
print("Last modified:",
      datetime.datetime.fromtimestamp(timestamp))'''

# Training and Evaluation

In [36]:
if RUN_TRAINING:

    results = run_training(
        PROJECT_ROOT, epochs=5, batch_size=64
    )

    print("Baseline Accuracy:", results["baseline"]["accuracy"])
    print("Classical Augmentation Accuracy:", results["classical_only"]["accuracy"])
    print("Synthetic + Classical Augmentation Accuracy:", results["synthetic_plus_classical"]["accuracy"])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]
100%|██████████| 18/18 [00:03<00:00,  5.52it/s]


Epoch 1/5, Loss: 3.0997


100%|██████████| 18/18 [00:02<00:00,  7.92it/s]


Epoch 2/5, Loss: 1.4395


...

100%|██████████| 18/18 [00:03<00:00,  4.56it/s]


Epoch 4/5, Loss: 0.6813


100%|██████████| 18/18 [00:03<00:00,  4.54it/s]

Epoch 5/5, Loss: 0.4536


Synthetic + Classical Augmentation Accuracy: 0.8312891796129735
Baseline Accuracy: 0.8250204415372036
Classical Augmentation Accuracy: 0.8277459798310166
Synthetic + Classical Augmentation Accuracy: 0.8312891796129735


In [39]:
# capture exact libraries version used in current environemnt in colab
# !pip freeze | grep -E "torch|torchvision|sentence-transformers|transformers|diffusers|accelerate|sentencepiece|scikit-learn|xformers|matplotlib|numpy|pillow|tqdm|bitsandbytes" > requirements_locked.txt